In [127]:
%%capture
%pip install pytube 
%pip install youtube-transcript-api==1.1.0
%pip install yt-dlp

In [128]:
import re
from pytube import YouTube
from langchain_core.tools import tool
from IPython.display import display, JSON
import yt_dlp
from typing import List, Dict
from langchain_core.messages import HumanMessage
from langchain_core.messages import ToolMessage
import json

# Suppress warnings
import warnings
warnings.filterwarnings("ignore")

# Suppress pytube errors
import logging
pytube_logger = logging.getLogger('pytube')
pytube_logger.setLevel(logging.ERROR)

# Suppress yt-dlp warnings
yt_dpl_logger = logging.getLogger('yt_dlp')
yt_dpl_logger.setLevel(logging.ERROR)

In [129]:
from dotenv import load_dotenv

In [130]:
load_dotenv()

True

In [131]:
from langchain_groq import ChatGroq

llm=ChatGroq(model="llama-3.1-8b-instant")

In [132]:
@tool
def extract_video_id(url: str) -> str:
    """
    Extracts the 11-character YouTube video ID from a URL.
    
    Args:
        url (str): A YouTube URL containing a video ID.

    Returns:
        str: Extracted video ID or error message if parsing fails.
    """
    
    # Regex pattern to match video IDs
    pattern = r'(?:v=|be/|embed/)([a-zA-Z0-9_-]{11})'
    match = re.search(pattern, url)
    return match.group(1) if match else "Error: Invalid YouTube URL"

In [133]:
extract_video_id.run("https://www.youtube.com/watch?v=hfIUstzHs9A")


'hfIUstzHs9A'

In [134]:
tools=[]
tools.append(extract_video_id)

In [135]:
tools

[StructuredTool(name='extract_video_id', description='Extracts the 11-character YouTube video ID from a URL.\n\nArgs:\n    url (str): A YouTube URL containing a video ID.\n\nReturns:\n    str: Extracted video ID or error message if parsing fails.', args_schema=<class 'langchain_core.utils.pydantic.extract_video_id'>, func=<function extract_video_id at 0x3085b1e40>)]

In [136]:
from youtube_transcript_api import YouTubeTranscriptApi



In [137]:
@tool
def fetch_transcript(video_id:str,language:str="en")->str:
    """
    Fetches the transcript of a YouTube video.
    
    Args:
        video_id (str): The YouTube video ID (e.g., "dQw4w9WgXcQ").
        language (str): Language code for the transcript (e.g., "en", "es").
    
    Returns:
        str: The transcript text or an error message.
    """

    try:
        yttapi=YouTubeTranscriptApi()
        transcript=yttapi.fetch(video_id,languages=[language])

        return " ".join([snippet.text for snippet in transcript.snippets])
    except Exception as e:
        return f"Error: {str(e)}"

In [138]:
fetch_transcript.run("hfIUstzHs9A")

'Over the past couple of months, large language models, or LLMs, such as chatGPT, have taken the world by storm. Whether it\'s writing poetry or helping plan your upcoming vacation, we are seeing a step change in the performance of AI and its potential to drive enterprise value. My name is Kate Soule. I\'m a senior manager of business strategy at IBM Research, and today I\'m going to give a brief overview of this new field of AI that\'s emerging and how it can be used in a business setting to drive value. Now, large language models are actually a part of a different class of models called foundation models. Now, the term "foundation models" was actually first coined by a team from Stanford when they saw that the field of AI was converging to a new paradigm. Where before AI applications were being built by training, maybe a library of different AI models, where each AI model was trained on very task-specific data to perform very specific task. They predicted that we were going to start 

In [139]:
tools.append(fetch_transcript)

In [140]:
from pytube import Search
from langchain.tools import tool
from typing import List, Dict

@tool
def search_youtube(query: str) -> List[Dict[str, str]]:
    """
    Search YouTube for videos matching the query.
    
    Args:
        query (str): The search term to look for on YouTube
        
    Returns:
        List of dictionaries containing video titles and IDs in format:
        [{'title': 'Video Title', 'video_id': 'abc123'}, ...]
        Returns error message if search fails
    """
    try:
        s = Search(query)
        return [
            {
                "title": yt.title,
                "video_id": yt.video_id,
                "url": f"https://youtu.be/{yt.video_id}"
            }
            for yt in s.results
        ]
    except Exception as e:
        return f"Error: {str(e)}"

In [141]:
search=search_youtube.run("Gen Ai")
search

[{'title': 'Gen AI Course | Gen AI Tutorial For Beginners',
  'video_id': 'd4yCWBGFCEs',
  'url': 'https://youtu.be/d4yCWBGFCEs'},
 {'title': 'Complete Agentic AI Course In 10 Hours- Langchain, Langgraph, RAG,Vectorless RAG, Guardrails,Evals',
  'video_id': 'rV3HJ4LEZ7k',
  'url': 'https://youtu.be/rV3HJ4LEZ7k'},
 {'title': 'Generative AI Explained In 5 Minutes | What Is GenAI? | Introduction To Generative AI | Simplilearn',
  'video_id': 'NRmAXDWJVnU',
  'url': 'https://youtu.be/NRmAXDWJVnU'},
 {'title': 'GenAI Essentials – Full Course for Beginners',
  'video_id': 'nJ25yl34Uqw',
  'url': 'https://youtu.be/nJ25yl34Uqw'},
 {'title': 'AI, Machine Learning, Deep Learning and Generative AI Explained',
  'video_id': 'qYNweeDHiyU',
  'url': 'https://youtu.be/qYNweeDHiyU'},
 {'title': '20 AI Concepts Explained in 40 Minutes',
  'video_id': 'OYvlznJ4IZQ',
  'url': 'https://youtu.be/OYvlznJ4IZQ'},
 {'title': 'Roadmap to Become a Generative AI Expert for Beginners in 2025',
  'video_id': '39zbC

In [142]:
tools.append(search_youtube)

In [143]:
tools

[StructuredTool(name='extract_video_id', description='Extracts the 11-character YouTube video ID from a URL.\n\nArgs:\n    url (str): A YouTube URL containing a video ID.\n\nReturns:\n    str: Extracted video ID or error message if parsing fails.', args_schema=<class 'langchain_core.utils.pydantic.extract_video_id'>, func=<function extract_video_id at 0x3085b1e40>),
 StructuredTool(name='fetch_transcript', description='Fetches the transcript of a YouTube video.\n\nArgs:\n    video_id (str): The YouTube video ID (e.g., "dQw4w9WgXcQ").\n    language (str): Language code for the transcript (e.g., "en", "es").\n\nReturns:\n    str: The transcript text or an error message.', args_schema=<class 'langchain_core.utils.pydantic.fetch_transcript'>, func=<function fetch_transcript at 0x3085b0220>),
 StructuredTool(name='search_youtube', description="Search YouTube for videos matching the query.\n\nArgs:\n    query (str): The search term to look for on YouTube\n\nReturns:\n    List of dictionaries

In [144]:
@tool
def get_full_metadata(url: str) -> dict:
    """Extract metadata given a YouTube URL, including title, views, duration, channel, likes, comments, and chapters."""
    with yt_dlp.YoutubeDL({'quiet': True, 'logger': yt_dpl_logger}) as ydl:
        info = ydl.extract_info(url, download=False)
        return {
            'title': info.get('title'),
            'views': info.get('view_count'),
            'duration': info.get('duration'),
            'channel': info.get('uploader'),
            'likes': info.get('like_count'),
            'comments': info.get('comment_count'),
            'chapters': info.get('chapters', [])
        }

In [145]:
meta_data=get_full_metadata.run("https://www.youtube.com/watch?v=T-D1OfcDW1M")
display(JSON(meta_data))

<IPython.core.display.JSON object>

In [146]:
tools.append(get_full_metadata)

In [147]:
tools

[StructuredTool(name='extract_video_id', description='Extracts the 11-character YouTube video ID from a URL.\n\nArgs:\n    url (str): A YouTube URL containing a video ID.\n\nReturns:\n    str: Extracted video ID or error message if parsing fails.', args_schema=<class 'langchain_core.utils.pydantic.extract_video_id'>, func=<function extract_video_id at 0x3085b1e40>),
 StructuredTool(name='fetch_transcript', description='Fetches the transcript of a YouTube video.\n\nArgs:\n    video_id (str): The YouTube video ID (e.g., "dQw4w9WgXcQ").\n    language (str): Language code for the transcript (e.g., "en", "es").\n\nReturns:\n    str: The transcript text or an error message.', args_schema=<class 'langchain_core.utils.pydantic.fetch_transcript'>, func=<function fetch_transcript at 0x3085b0220>),
 StructuredTool(name='search_youtube', description="Search YouTube for videos matching the query.\n\nArgs:\n    query (str): The search term to look for on YouTube\n\nReturns:\n    List of dictionaries

In [148]:
@tool
def get_thumbnails(url: str) -> List[Dict]:
    """
    Get available thumbnails for a YouTube video using its URL.
    
    Args:
        url (str): YouTube video URL (any format)
        
    Returns:
        List of dictionaries with thumbnail URLs and resolutions in YouTube's native order
    """
    
    try:
        with yt_dlp.YoutubeDL({'quiet': True, 'logger': yt_dpl_logger}) as ydl:
            info = ydl.extract_info(url, download=False)
            
            thumbnails = []
            for t in info.get('thumbnails', []):
                if 'url' in t:
                    thumbnails.append({
                        "url": t['url'],
                        "width": t.get('width'),
                        "height": t.get('height'),
                        "resolution": f"{t.get('width', '')}x{t.get('height', '')}".strip('x')
                    })
            
            return thumbnails

    except Exception as e:
        return [{"error": f"Failed to get thumbnails: {str(e)}"}]

In [149]:
thumbnails=get_thumbnails.run("https://www.youtube.com/watch?v=qWHaMrR5WHQ")

display(JSON(thumbnails))

<IPython.core.display.JSON object>

In [150]:
tools.append(get_thumbnails)

In [151]:
llm_with_tools=llm.bind_tools(tools)

In [152]:
for tool in tools:
    schema = {
   "name": tool.name,
   "description": tool.description,
   "parameters": tool.args_schema.schema() if tool.args_schema else {},
   "return": tool.return_type if hasattr(tool, "return_type") else None}
    display(JSON(schema))

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

In [153]:
query = "I want to summarize youtube video: https://www.youtube.com/watch?v=T-D1OfcDW1M in english"

In [154]:
messages=[HumanMessage(content=query)]
messages

[HumanMessage(content='I want to summarize youtube video: https://www.youtube.com/watch?v=T-D1OfcDW1M in english', additional_kwargs={}, response_metadata={})]

In [155]:
response=llm_with_tools.invoke(messages)
response

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'tqwwnxqen', 'function': {'arguments': '{"url":"https://www.youtube.com/watch?v=T-D1OfcDW1M"}', 'name': 'get_full_metadata'}, 'type': 'function'}, {'id': 'qb15ax09a', 'function': {'arguments': '{"language":"en","video_id":"T-D1OfcDW1M"}', 'name': 'fetch_transcript'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 58, 'prompt_tokens': 910, 'total_tokens': 968, 'completion_time': 0.11196375, 'completion_tokens_details': None, 'prompt_time': 0.174787667, 'prompt_tokens_details': None, 'queue_time': 0.063399179, 'total_time': 0.286751417}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--019e4f96-b858-7491-b098-c4ef8643ecf9-0', tool_calls=[{'name': 'get_full_metadata', 'args': {'url': 'https://www.youtube.com/watch?v=T-D1OfcDW1M'}, 'id': 'tqwwnxqen', 'type': 'tool_call'}, {'name': '

In [156]:
messages.append(response)

In [157]:
tool_mapping = {
    "get_thumbnails" : get_thumbnails,
    "extract_video_id": extract_video_id,
    "fetch_transcript": fetch_transcript,
    "search_youtube": search_youtube,
    "get_full_metadata": get_full_metadata
}

In [158]:
tool_calls_1 = response.tool_calls
display(JSON(tool_calls_1))

<IPython.core.display.JSON object>

In [159]:
tool_name=tool_calls_1[0]['name']
tool_name

'get_full_metadata'

In [160]:
tool_call_id =tool_calls_1[0]['id']
print(tool_call_id)

tqwwnxqen


In [161]:
args=tool_calls_1[0]['args']
print(args)

{'url': 'https://www.youtube.com/watch?v=T-D1OfcDW1M'}


In [162]:
my_tool=tool_mapping[tool_calls_1[0]['name']]

In [163]:
my_tool

StructuredTool(name='get_full_metadata', description='Extract metadata given a YouTube URL, including title, views, duration, channel, likes, comments, and chapters.', args_schema=<class 'langchain_core.utils.pydantic.get_full_metadata'>, func=<function get_full_metadata at 0x30a2c9b20>)

In [164]:
video_id=my_tool.invoke(tool_calls_1[0]['args'])

video_id

{'title': 'What is Retrieval-Augmented Generation (RAG)?',
 'views': 1829519,
 'duration': 395,
 'channel': 'IBM Technology',
 'likes': 43035,
 'comments': 960,
 'chapters': [{'start_time': 0.0, 'title': 'Introduction', 'end_time': 18.0},
  {'start_time': 18.0, 'title': 'What is RAG', 'end_time': 42.0},
  {'start_time': 42.0, 'title': 'An anecdote', 'end_time': 82.0},
  {'start_time': 82.0, 'title': 'Two problems', 'end_time': 138.0},
  {'start_time': 138.0, 'title': 'Large language models', 'end_time': 263.0},
  {'start_time': 263.0, 'title': 'How does RAG help', 'end_time': 395}]}

In [165]:
messages.append(ToolMessage(content = video_id, tool_call_id = tool_calls_1[0]['id']))

In [166]:
response_2=llm_with_tools.invoke(messages)

In [167]:
messages.append(response_2)

In [168]:
tool_calls_2 = response_2.tool_calls
tool_calls_2

[]

In [169]:
fetch_transcript_tool_output = tool_mapping[tool_calls_2[0]['name']].invoke(tool_calls_2[0]['args'])
fetch_transcript_tool_output

IndexError: list index out of range

In [ ]:
messages.append(ToolMessage(content = fetch_transcript_tool_output, tool_call_id = tool_calls_2[0]['id']))

In [ ]:
summary = llm_with_tools.invoke(messages)

summary

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'jwmnkht6t', 'function': {'arguments': '{"language":"en","video_id":"T-D1OfcDW1M"}', 'name': 'fetch_transcript'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 1421, 'total_tokens': 1451, 'completion_time': 0.025535685, 'completion_tokens_details': None, 'prompt_time': 0.36667281, 'prompt_tokens_details': None, 'queue_time': 0.118236917, 'total_time': 0.392208495}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--019e4f8d-6bbd-7000-abd2-340052f869c0-0', tool_calls=[{'name': 'fetch_transcript', 'args': {'language': 'en', 'video_id': 'T-D1OfcDW1M'}, 'id': 'jwmnkht6t', 'type': 'tool_call'}], usage_metadata={'input_tokens': 1421, 'output_tokens': 30, 'total_tokens': 1451})

In [ ]:
# Define the processing steps
def execute_tool(tool_call):
    """Execute single tool call and return ToolMessage"""
    try:
        result = tool_mapping[tool_call["name"]].invoke(tool_call["args"])
        return ToolMessage(
            content=str(result),
            tool_call_id=tool_call["id"]
        )
    except Exception as e:
        return ToolMessage(
            content=f"Error: {str(e)}",
            tool_call_id=tool_call["id"]
        )

        

In [ ]:
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

In [ ]:
summarization_chain = (
    # Start with initial query
    RunnablePassthrough.assign(
        messages=lambda x: [HumanMessage(content=x["query"])]
    )
    # First LLM call (extract video ID)
    | RunnablePassthrough.assign(
        ai_response=lambda x: llm_with_tools.invoke(x["messages"])
    )
    # Process first tool call
    | RunnablePassthrough.assign(
        tool_messages=lambda x: [
            execute_tool(tc) for tc in x["ai_response"].tool_calls
        ]
    )
    # Update message history
    | RunnablePassthrough.assign(
        messages=lambda x: x["messages"] + [x["ai_response"]] + x["tool_messages"]
    )
    # Second LLM call (fetch transcript)
    | RunnablePassthrough.assign(
        ai_response2=lambda x: llm_with_tools.invoke(x["messages"])
    )
    # Process second tool call
    | RunnablePassthrough.assign(
        tool_messages2=lambda x: [
            execute_tool(tc) for tc in x["ai_response2"].tool_calls
        ]
    )
    # Final message update
    | RunnablePassthrough.assign(
        messages=lambda x: x["messages"] + [x["ai_response2"]] + x["tool_messages2"]
    )
    # Generate final summary
    | RunnablePassthrough.assign(
        summary=lambda x: llm_with_tools.invoke(x["messages"]).content
    )
    # Return just the summary text
    | RunnableLambda(lambda x: x["summary"])
)

In [170]:
# Usage
result = summarization_chain.invoke({
    "query": "Summarize this YouTube video: https://www.youtube.com/watch?v=1bUy-1hGZpI"
})

print("Video Summary:\n", result)

Video Summary:
  However, based on the metadata provided, we can infer that the video is about LangChain, a technology developed by IBM.
